In [ ]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Joint Refinement: Si, Bragg + PDF

This example demonstrates a joint refinement of the Si crystal
structure combining Bragg diffraction and pair distribution function
(PDF) analysis. The Bragg experiment uses time-of-flight neutron
powder diffraction data from SEPD at Argonne, while the PDF
experiment uses data from NOMAD at SNS. A single shared Si structure
is refined simultaneously against both datasets.

## 🛠️ Import Library

In [ ]:
from easydiffraction import ExperimentFactory
from easydiffraction import Project
from easydiffraction import StructureFactory
from easydiffraction import download_data

## 🧩 Define Structure

A single Si structure is shared between the Bragg and PDF
experiments. Structural parameters refined against both datasets
simultaneously.

### Create Structure

In [ ]:
structure = StructureFactory.from_scratch(name='si')

### Set Space Group

In [ ]:
structure.space_group.name_h_m = 'F d -3 m'
structure.space_group.it_coordinate_system_code = '1'

### Set Unit Cell

In [ ]:
structure.cell.length_a = 5.42

### Set Atom Sites

In [ ]:
structure.atom_sites.create(
    label='Si',
    type_symbol='Si',
    fract_x=0,
    fract_y=0,
    fract_z=0,
    adp_iso=0.2,
)

## 🔬 Define Experiments

Two experiments are defined: one for Bragg diffraction and one for
PDF analysis. Both are linked to the same Si structure.

### Experiment 1: Bragg (SEPD, TOF)

#### Download Data

In [ ]:
bragg_data_path = download_data(id=7, destination='data')

#### Create Experiment

In [ ]:
bragg_expt = ExperimentFactory.from_data_path(
    name='sepd', data_path=bragg_data_path, beam_mode='time-of-flight'
)

#### Set Instrument

In [ ]:
bragg_expt.instrument.setup_twotheta_bank = 144.845
bragg_expt.instrument.calib_d_to_tof_offset = -9.2
bragg_expt.instrument.calib_d_to_tof_linear = 7476.91
bragg_expt.instrument.calib_d_to_tof_quad = -1.54

#### Set Peak Profile

In [ ]:
bragg_expt.peak.type = 'jorgensen'
bragg_expt.peak.broad_gauss_sigma_0 = 5.0
bragg_expt.peak.broad_gauss_sigma_1 = 45.0
bragg_expt.peak.broad_gauss_sigma_2 = 1.0
bragg_expt.peak.exp_decay_beta_0 = 0.04221
bragg_expt.peak.exp_decay_beta_1 = 0.00946
bragg_expt.peak.exp_rise_alpha_0 = 0.0
bragg_expt.peak.exp_rise_alpha_1 = 0.5971

#### Set Background

In [ ]:
bragg_expt.background.type = 'line-segment'
for x in range(0, 35000, 5000):
    bragg_expt.background.create(id=str(x), x=x, y=200)

#### Set Linked Phases

In [ ]:
bragg_expt.linked_phases.create(id='si', scale=13.0)

### Experiment 2: PDF (NOMAD, TOF)

#### Download Data

In [ ]:
pdf_data_path = download_data(id=5, destination='data')

#### Create Experiment

In [ ]:
pdf_expt = ExperimentFactory.from_data_path(
    name='nomad',
    data_path=pdf_data_path,
    beam_mode='time-of-flight',
    scattering_type='total',
)

#### Set Peak Profile (PDF Parameters)

In [ ]:
pdf_expt.peak.damp_q = 0.02
pdf_expt.peak.broad_q = 0.02
pdf_expt.peak.cutoff_q = 35.0
pdf_expt.peak.sharp_delta_1 = 0.001
pdf_expt.peak.sharp_delta_2 = 4.0
pdf_expt.peak.damp_particle_diameter = 0

#### Set Linked Phases

In [ ]:
pdf_expt.linked_phases.create(id='si', scale=1.0)

## 📦 Define Project

The project object manages the shared structure, both experiments,
and the analysis.

### Create Project

In [ ]:
project = Project()

### Add Structure

In [ ]:
project.structures.add(structure)

### Add Experiments

In [ ]:
project.experiments.add(bragg_expt)
project.experiments.add(pdf_expt)

## 🚀 Perform Analysis

This section shows the joint analysis process. The calculator is
auto-resolved per experiment: CrysPy for Bragg, PDFfit for PDF.

### Set Fit Mode and Weights

In [ ]:
project.analysis.fitting_mode.type = 'joint'
project.analysis.joint_fit.create(experiment_id='sepd', weight=0.7)
project.analysis.joint_fit.create(experiment_id='nomad', weight=0.3)

### Display Structure

In [ ]:
project.display.structure(struct_name='si')

### Display Pattern (Before Fit)

In [ ]:
project.display.pattern(expt_name='sepd')

In [ ]:
project.display.pattern(expt_name='nomad')

### Set Free Parameters

Shared structural parameters are refined against both datasets
simultaneously.

In [ ]:
structure.cell.length_a.free = True
structure.atom_sites['Si'].adp_iso.free = True

Bragg experiment parameters.

In [ ]:
bragg_expt.linked_phases['si'].scale.free = True
bragg_expt.instrument.calib_d_to_tof_offset.free = True
bragg_expt.peak.broad_gauss_sigma_0.free = True
bragg_expt.peak.broad_gauss_sigma_1.free = True
bragg_expt.peak.broad_gauss_sigma_2.free = True
for point in bragg_expt.background:
    point.y.free = True

PDF experiment parameters.

In [ ]:
pdf_expt.linked_phases['si'].scale.free = True
pdf_expt.peak.damp_q.free = True
pdf_expt.peak.broad_q.free = True
pdf_expt.peak.sharp_delta_1.free = True
pdf_expt.peak.sharp_delta_2.free = True

### Display Free Parameters

In [ ]:
project.display.parameters.free()

### Run Fitting

In [ ]:
project.analysis.fit()
project.display.fit.results()
project.display.fit.correlations()

### Display Pattern (After Fit)

In [ ]:
project.display.pattern(expt_name='sepd')

In [ ]:
project.display.pattern(expt_name='nomad')

## 💾 Save Project

In [ ]:
project.save_as(dir_path='projects/ed_16_si_bragg_pdf')